# Demo Modul 4: Regularisasi dan Hyperparameter Tuning

**Durasi sesi:** 120 menit  
**Kasus:** Fashion-MNIST dengan data latih sengaja dikecilkan supaya overfitting terlihat.

Modul 3 mencari cara melatih lebih cepat. Notebook ini menjawab pertanyaan berikutnya: bagaimana memastikan model bekerja pada data yang belum pernah dilihat.

## Capaian demo

Setelah demo, praktikan dapat:

1. membuat overfitting terlihat dan mengukurnya sebagai *generalization gap*;
2. menguji dropout, weight decay, dan early stopping secara terpisah;
3. menyusun ruang pencarian dan menjalankan random search dengan anggaran tetap;
4. memilih konfigurasi dari validation set; dan
5. menjaga test set sampai satu evaluasi terakhir.

In [ ]:
import copy
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import FashionMNIST

SEED = 42            # seed data dan inisialisasi
SEED_CARI = 7        # seed pengambilan sampel random search
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE)})

## 1. Data: kecil di latih, cukup di validasi

Overfitting tidak muncul sendiri pada data besar. Protokol modul ini sengaja memakai $3\,000$ citra latih agar gejalanya terlihat dalam waktu sesi.

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

latih_penuh = FashionMNIST(root=DATA_ROOT, train=True, download=False,
                           transform=transforms.ToTensor())
uji_resmi = FashionMNIST(root=DATA_ROOT, train=False, download=False,
                         transform=transforms.ToTensor())

X_penuh = latih_penuh.data.float().unsqueeze(1) / 255.0
y_penuh = latih_penuh.targets

idx_latih, idx_val = train_test_split(
    np.arange(len(y_penuh)), train_size=3_000, test_size=3_000,
    stratify=y_penuh.numpy(), random_state=SEED)

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]

MEAN, STD = X_latih.mean().item(), X_latih.std().item()
normalkan = lambda t: (t - MEAN) / STD

ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)

# Catatan: ds_uji SENGAJA belum dibuat. Ia baru muncul pada sel terakhir.
print(f'latih {len(ds_latih)}  validasi {len(ds_val)}')
print('per kelas di latih:', torch.bincount(y_latih).tolist())

**Pemeriksaan:** $300$ citra per kelas pada subset latih. Perhatikan bahwa `ds_uji` belum dibuat sama sekali — itu bagian dari protokol anti-kebocoran, bukan kelalaian.

## 2. Model dan satu fungsi pelatihan

Model menerima `hidden` dan `dropout` sebagai argumen, sehingga baseline dan seluruh trial memakai kode yang sama.

In [ ]:
BATCH, EPOCH = 128, 30

def buat_model(hidden=512, dropout=0.0):
    seed_everything(SEED)
    lapis = [nn.Flatten(), nn.Linear(28 * 28, hidden), nn.ReLU()]
    if dropout > 0:
        lapis.append(nn.Dropout(dropout))
    lapis += [nn.Linear(hidden, hidden), nn.ReLU()]
    if dropout > 0:
        lapis.append(nn.Dropout(dropout))
    lapis.append(nn.Linear(hidden, 10))
    return nn.Sequential(*lapis).to(DEVICE)

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    model.eval()                       # mematikan dropout
    kriteria = nn.CrossEntropyLoss(reduction='sum')
    total, benar = 0.0, 0
    for xb, yb in loader(ds, batch, False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total += kriteria(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total / len(ds), benar / len(ds)

print('parameter baseline:', sum(p.numel() for p in buat_model().parameters()))

**Pemeriksaan:** baseline berjumlah $669\,706$ parameter melawan $3\,000$ contoh latih — lebih dari dua ratus parameter per contoh. Perbandingan inilah yang membuat overfitting hampir pasti terjadi.

In [ ]:
def jalankan(hidden=512, dropout=0.0, weight_decay=0.0, lr=1e-3,
             epoch=EPOCH, sabar=None, label='baseline'):
    model = buat_model(hidden, dropout)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    kriteria = nn.CrossEntropyLoss()
    dl = loader(ds_latih, BATCH, True)

    riwayat = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    terbaik = {'val_loss': float('inf'), 'epoch': 0, 'bobot': None}
    mulai, berhenti_di = time.perf_counter(), epoch

    for ep in range(1, epoch + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            kriteria(model(xb), yb).backward()
            opt.step()

        tl, ta = evaluasi(model, ds_latih)
        vl, va = evaluasi(model, ds_val)
        for k, v in zip(riwayat, (tl, vl, ta, va)):
            riwayat[k].append(v)

        if vl < terbaik['val_loss']:
            terbaik = {'val_loss': vl, 'epoch': ep,
                       'bobot': copy.deepcopy(model.state_dict())}
        elif sabar is not None and ep - terbaik['epoch'] >= sabar:
            berhenti_di = ep
            break

    model.load_state_dict(terbaik['bobot'])      # selalu pakai bobot terbaik
    vl, va = evaluasi(model, ds_val)
    tl, _ = evaluasi(model, ds_latih)
    return model, riwayat, {
        'run_id': label, 'seed': SEED, 'hidden': hidden, 'dropout': dropout,
        'weight_decay': weight_decay, 'learning_rate': lr,
        'parameter': sum(p.numel() for p in model.parameters()),
        'epoch_terbaik': terbaik['epoch'], 'epoch_berhenti': berhenti_di,
        'train_loss': tl, 'val_loss': vl, 'val_acc': va, 'gap': vl - tl,
        'runtime_s': time.perf_counter() - mulai,
    }

## 3. Baseline: membuat overfitting terlihat

In [ ]:
_, riwayat_base, catatan_base = jalankan(label='baseline')
print(pd.Series(catatan_base))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ep = range(1, len(riwayat_base['train_loss']) + 1)
ax[0].plot(ep, riwayat_base['train_loss'], label='train')
ax[0].plot(ep, riwayat_base['val_loss'], label='validation')
ax[0].axvline(catatan_base['epoch_terbaik'], ls='--', c='gray')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, riwayat_base['train_acc'], label='train')
ax[1].plot(ep, riwayat_base['val_acc'], label='validation')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].grid(alpha=.3)
fig.suptitle('Baseline tanpa regularisasi: validation loss naik setelah titik terendah')
plt.tight_layout(); plt.show()

Perhatikan bentuknya: train loss terus turun sementara validation loss berbalik naik setelah epoch terbaik. Garis putus-putus menandai epoch itu. Akurasi latih yang tinggi di sini justru gejala, bukan prestasi.

## 4. Tiga strategi, diuji terpisah

Setiap run hanya mengubah **satu** hal dibanding baseline.

In [ ]:
hasil = [catatan_base]
for label, kwargs in [
    ('dropout-0.5', dict(dropout=0.5)),
    ('wd-1e-3', dict(weight_decay=1e-3)),
    ('early-stopping', dict(sabar=5)),
]:
    _, _, catatan = jalankan(label=label, **kwargs)
    hasil.append(catatan)

pd.DataFrame(hasil)[['run_id', 'epoch_terbaik', 'epoch_berhenti', 'train_loss',
                     'val_loss', 'val_acc', 'gap', 'runtime_s']]

Bacalah kolom `gap` dan `val_loss` bersama-sama. Strategi yang paling menurunkan gap belum tentu yang paling menurunkan validation loss — dan yang dipakai untuk memilih model adalah validation loss.

## 5. Ruang pencarian dan random search

Ruangnya berisi $3^4 = 81$ kombinasi; anggaran kita $12$ trial, yaitu sekitar $15\%$ ruang. Seed pencarian dicatat supaya hasilnya dapat diulang.

In [ ]:
ruang = {'hidden': [64, 128, 256],
         'dropout': [0.0, 0.2, 0.5],
         'weight_decay': [0.0, 1e-4, 1e-3],
         'learning_rate': [1e-4, 3e-4, 1e-3]}

semua_kombinasi = [(h, d, w, l) for h in ruang['hidden'] for d in ruang['dropout']
                   for w in ruang['weight_decay'] for l in ruang['learning_rate']]
rng = np.random.default_rng(SEED_CARI)
terpilih = rng.choice(len(semua_kombinasi), size=12, replace=False)
print(f'ukuran ruang {len(semua_kombinasi)}, anggaran 12 trial '
      f'({12 / len(semua_kombinasi):.1%})')

trial = []
for n, i in enumerate(terpilih, start=1):
    h, d, w, l = semua_kombinasi[i]
    _, _, catatan = jalankan(hidden=h, dropout=d, weight_decay=w, lr=l,
                             sabar=5, label=f'trial-{n:02d}')
    trial.append(catatan)

tabel = pd.DataFrame(trial).sort_values('val_loss')
tabel[['run_id', 'hidden', 'dropout', 'weight_decay', 'learning_rate',
       'parameter', 'epoch_terbaik', 'val_loss', 'val_acc', 'gap']].head(12)

Periksa kolom `parameter` pada tiga trial teratas. Model dengan parameter terbanyak sering **tidak** berada di puncak — pada data latih yang kecil, kapasitas besar justru memperbesar gap.

## 6. Evaluasi akhir: test set dibuka satu kali

Baru pada tahap ini `ds_uji` dibuat. Sebelum sel ini dijalankan, seluruh keputusan sudah terkunci.

In [ ]:
juara = tabel.iloc[0]
print('konfigurasi terpilih:', juara[['hidden', 'dropout', 'weight_decay',
                                      'learning_rate']].to_dict())

model_final, _, catatan_final = jalankan(
    hidden=int(juara['hidden']), dropout=float(juara['dropout']),
    weight_decay=float(juara['weight_decay']), lr=float(juara['learning_rate']),
    sabar=5, label='final')

# ---- test loader baru dibuat di sini, dan hanya dipakai sekali ----
X_uji = uji_resmi.data.float().unsqueeze(1) / 255.0
ds_uji = TensorDataset(normalkan(X_uji), uji_resmi.targets)
test_loss, test_acc = evaluasi(model_final, ds_uji)

print(f"validation loss : {catatan_final['val_loss']:.4f}")
print(f"test loss       : {test_loss:.4f}")
print(f"test accuracy   : {test_acc:.4f}")
print(f"gap baseline    : {catatan_base['gap']:.4f}  ->  final: {catatan_final['gap']:.4f}")

## Exit ticket

1. Apa bukti angka yang Anda pakai untuk menyatakan baseline overfit?
2. Mengapa `model.eval()` wajib dipanggil sebelum menghitung metrik ketika model memakai dropout?
3. Mengapa test loader sengaja baru dibuat pada sel terakhir?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb` — baseline, tiga strategi terpisah, dua belas trial random search, lalu satu evaluasi test pada konfigurasi terpilih.